<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/7-lambda-tools-data-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lambda Tools for Data Analysis with NumPy and Pandas

This notebook demonstrates how to create **Lambda Tools** that leverage NumPy and Pandas for data analysis tasks. Lambda tools enable agents to perform computations that would be difficult or impossible with pure LLM reasoning.

You'll learn how to:
1. Create Lambda tools that use NumPy and Pandas
2. Pass structured data (JSON/CSV) to tools and receive computed results
3. Build a Statistical Analyzer for descriptive statistics and correlations
4. Build a Trend Analyzer for time-series analysis
5. Combine these tools with an agent for comprehensive data analysis

## Why Lambda Tools for Data Analysis?

LLMs are powerful at reasoning and language tasks, but they have limitations:

- **Numerical precision**: LLMs can make arithmetic errors, especially with large datasets
- **Statistical computations**: Calculating correlations, standard deviations, or regressions requires exact math
- **Data transformations**: Normalizing, pivoting, or cleaning data needs deterministic operations
- **Scalability**: Processing thousands of rows requires actual computation, not token prediction

Lambda tools solve this by executing real Python code with libraries like NumPy and Pandas, giving agents access to precise, scalable data analysis capabilities.

## Getting Started

This notebook assumes you've completed Notebooks 1-6:
- Notebook 1: Created corpora
- Notebook 2: Ingested data
- Notebook 3: Queried data
- Notebook 4: Created agents and sessions
- Notebook 5: Built multi-agent workflows with Lambda tools
- Notebook 6: Worked with file artifacts

Now we'll create sophisticated Lambda tools for data analysis using NumPy and Pandas.

## Setup

In [1]:
import os
import json
from datetime import datetime

import requests

# Get credentials from environment variables
api_key = os.environ['VECTARA_API_KEY']

# Base API URL
BASE_URL = "https://api.vectara.io/v2"

# Common headers
headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

print("Setup complete!")

Setup complete!


In [2]:
# Load the shared helpers (delete_and_create_agent / delete_and_create_tool).
# vectara_utils.py sits next to this notebook in the repo; Colab fetches it on
# first run so the same notebook works locally and in a fresh Colab runtime.
try:
    from vectara_utils import (
        delete_and_create_agent,
        delete_and_create_tool,
        find_agents_by_name,
    )
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import (
        delete_and_create_agent,
        delete_and_create_tool,
        find_agents_by_name,
    )

import vectara_utils
vectara_utils.configure(BASE_URL, headers)


## Step 1: Create a Statistical Analyzer Lambda Tool

This tool uses Pandas to compute descriptive statistics on tabular data. It can calculate:
- Basic stats: mean, median, std, min, max, percentiles
- Correlations between numeric columns
- Value counts for categorical columns

The tool accepts JSON data (as a string) and returns computed statistics.

In [3]:
import time

# Cleanup: a Data Analyst agent from a prior run still references the lambda tools
# we're about to (re)create, which blocks their DELETE with "tool is being used in
# different configurations". Tear down the agent first, then wait for the async
# DELETE to clear before recreating the tools below.
for _agent in find_agents_by_name("Data Analyst"):
    requests.delete(f"{BASE_URL}/agents/{_agent['key']}", headers=headers)
    print(f"Deleted prior Data Analyst agent: {_agent['key']}")

# Wait for the async DELETE to clear so the (re)created tools below aren't
# blocked by a still-living agent's reference to them.
vectara_utils.wait_for_agent_gone("Data Analyst")

statistical_analyzer_code = '''
import json
import pandas as pd
import numpy as np

def process(
    data: str,
    columns: str = "",
    operations: str = "describe"
) -> dict:
    """
    Compute statistical analysis on tabular data using Pandas.
    
    Args:
        data: JSON string containing the data (list of dicts or dict of lists)
        columns: Comma-separated column names to analyze (empty = all numeric columns)
        operations: Comma-separated operations: describe, correlation, value_counts, percentiles
    
    Returns:
        Dictionary with computed statistics
    """
    results = {"success": True, "statistics": {}}
    
    # Parse the input data
    try:
        parsed_data = json.loads(data)
        df = pd.DataFrame(parsed_data)
    except Exception as e:
        return {"success": False, "error": f"Failed to parse data: {str(e)}"}
    
    # Filter columns if specified
    if columns:
        col_list = [c.strip() for c in columns.split(",")]
        valid_cols = [c for c in col_list if c in df.columns]
        if not valid_cols:
            return {"success": False, "error": f"None of the specified columns found. Available: {list(df.columns)}"}
        df_analysis = df[valid_cols]
    else:
        df_analysis = df.select_dtypes(include=[np.number])
    
    # Parse operations
    ops = [op.strip().lower() for op in operations.split(",")]
    
    # Execute requested operations
    if "describe" in ops:
        desc = df_analysis.describe()
        results["statistics"]["describe"] = desc.to_dict()
    
    if "correlation" in ops:
        numeric_df = df_analysis.select_dtypes(include=[np.number])
        if len(numeric_df.columns) >= 2:
            corr = numeric_df.corr()
            results["statistics"]["correlation"] = corr.to_dict()
        else:
            results["statistics"]["correlation"] = "Need at least 2 numeric columns"
    
    if "value_counts" in ops:
        value_counts = {}
        for col in df_analysis.columns:
            vc = df_analysis[col].value_counts().head(10)
            value_counts[col] = vc.to_dict()
        results["statistics"]["value_counts"] = value_counts
    
    if "percentiles" in ops:
        percentiles = {}
        numeric_df = df_analysis.select_dtypes(include=[np.number])
        for col in numeric_df.columns:
            percentiles[col] = {
                "p10": float(numeric_df[col].quantile(0.10)),
                "p25": float(numeric_df[col].quantile(0.25)),
                "p50": float(numeric_df[col].quantile(0.50)),
                "p75": float(numeric_df[col].quantile(0.75)),
                "p90": float(numeric_df[col].quantile(0.90)),
                "p99": float(numeric_df[col].quantile(0.99))
            }
        results["statistics"]["percentiles"] = percentiles
    
    # Add metadata
    results["metadata"] = {
        "rows": len(df),
        "columns_analyzed": list(df_analysis.columns),
        "operations_performed": ops
    }
    
    return results
'''

statistical_analyzer_config = {
    "type": "lambda",
    "language": "python",
    "name": "statistical_analyzer",
    "title": "Statistical Analyzer",
    "description": """Compute statistical analysis on tabular data using Pandas and NumPy.
    
Pass data as a JSON string (list of dicts). Specify columns to analyze (comma-separated) or leave empty for all numeric columns.

Available operations (comma-separated):
- describe: Basic stats (mean, std, min, max, quartiles)
- correlation: Correlation matrix between numeric columns  
- value_counts: Top 10 most frequent values per column
- percentiles: p10, p25, p50, p75, p90, p99 for numeric columns

Example: operations="describe,correlation" will return both descriptive stats and correlations.""",
    "code": statistical_analyzer_code
}

statistical_analyzer_id = delete_and_create_tool(statistical_analyzer_config, "statistical_analyzer")


Created tool 'statistical_analyzer' (id: tol_6330)


## Step 2: Create a Trend Analyzer Lambda Tool

This tool uses NumPy for time-series analysis:
- Moving averages (3 and 7 period windows)
- Growth rates (period-over-period and total)
- Trend detection (linear regression slope and direction)

In [4]:
trend_analyzer_code = '''
import json
import numpy as np

def process(
    data: str,
    date_column: str,
    value_column: str,
    analysis_type: str = "all"
) -> dict:
    """
    Analyze trends in time-series data using NumPy.

    Args:
        data: JSON string containing the time-series data
        date_column: Name of the date/time column (must be in ISO format, e.g. YYYY-MM-DD)
        value_column: Name of the numeric value column to analyze
        analysis_type: Comma-separated: moving_average, growth_rate, trend, all

    Returns:
        Dictionary with trend analysis results
    """
    results = {"success": True, "analysis": {}}

    try:
        parsed_data = json.loads(data)
    except Exception as e:
        return {"success": False, "error": "Failed to parse data: " + str(e)}

    if not parsed_data:
        return {"success": False, "error": "Empty dataset"}

    keys = list(parsed_data[0].keys())
    if date_column not in keys:
        return {"success": False, "error": "Date column not found. Available: " + str(keys)}
    if value_column not in keys:
        return {"success": False, "error": "Value column not found. Available: " + str(keys)}

    sorted_data = sorted(parsed_data, key=lambda row: row[date_column])
    dates = [row[date_column] for row in sorted_data]
    values = [float(row[value_column]) for row in sorted_data]
    n = len(values)

    analyses = [a.strip().lower() for a in analysis_type.split(",")]
    do_all = "all" in analyses

    if do_all or "moving_average" in analyses:
        ma_results = {}
        for window in [3, 7]:
            if n >= window:
                ma = []
                for i in range(window - 1, n):
                    avg = sum(values[i - window + 1:i + 1]) / window
                    ma.append(round(avg, 2))
                ma_results["ma_" + str(window)] = {
                    "latest": ma[-1],
                    "values": ma[-10:]
                }
        results["analysis"]["moving_averages"] = ma_results

    if do_all or "growth_rate" in analyses:
        growth = {}
        if n >= 2:
            pct_changes = []
            for i in range(1, n):
                if values[i - 1] != 0:
                    pct_changes.append((values[i] - values[i - 1]) / values[i - 1])
            if pct_changes:
                growth["period_over_period"] = {
                    "latest": round(pct_changes[-1], 4),
                    "mean": round(sum(pct_changes) / len(pct_changes), 4),
                }
        if values[0] != 0:
            growth["total_growth"] = round((values[-1] - values[0]) / values[0], 4)
        results["analysis"]["growth_rates"] = growth

    if do_all or "trend" in analyses:
        x = list(range(n))
        mean_x = sum(x) / n
        mean_y = sum(values) / n
        num = sum((x[i] - mean_x) * (values[i] - mean_y) for i in range(n))
        den = sum((x[i] - mean_x) ** 2 for i in range(n))
        slope = num / den if den != 0 else 0
        intercept = mean_y - slope * mean_x
        ss_res = sum((values[i] - (slope * x[i] + intercept)) ** 2 for i in range(n))
        ss_tot = sum((values[i] - mean_y) ** 2 for i in range(n))
        r_squared = 1 - (ss_res / ss_tot) if ss_tot != 0 else 0

        if mean_y != 0 and slope > 0.01 * mean_y:
            direction = "upward"
        elif mean_y != 0 and slope < -0.01 * mean_y:
            direction = "downward"
        else:
            direction = "flat"

        results["analysis"]["trend"] = {
            "slope": round(slope, 2),
            "intercept": round(intercept, 2),
            "r_squared": round(r_squared, 4),
            "direction": direction,
            "slope_per_period_pct": round(slope / mean_y * 100, 2) if mean_y != 0 else 0
        }

    results["summary"] = {
        "start_date": dates[0],
        "end_date": dates[-1],
        "n_periods": n,
        "start_value": values[0],
        "end_value": values[-1],
        "min": min(values),
        "max": max(values),
        "mean": round(sum(values) / n, 2)
    }

    return results
'''

trend_analyzer_config = {
    "type": "lambda",
    "language": "python",
    "name": "trend_analyzer",
    "title": "Trend Analyzer",
    "description": """Analyze trends in time-series data using NumPy.

Pass time-series data as JSON, specifying the date column and value column to analyze.
Date column must be in ISO format (YYYY-MM-DD) for correct sorting.

Available analysis types (comma-separated):
- moving_average: Simple moving averages (3 and 7 period windows)
- growth_rate: Period-over-period and total growth
- trend: Linear regression to detect trend direction, slope, and R-squared
- all: Perform all analyses (default)

Returns trend direction (upward/downward/flat), growth metrics, and moving averages.""",
    "code": trend_analyzer_code
}

trend_analyzer_id = delete_and_create_tool(trend_analyzer_config, "trend_analyzer")

Created tool 'trend_analyzer' (id: tol_6331)


## Step 3: Create a Data Analyst Agent

Now we'll create an agent equipped with both data analysis Lambda tools. This agent can:
- Analyze datasets with statistical methods
- Identify trends in time-series data

In [5]:
# Delete previous agent/session if re-running this cell
if 'session_key' in dir() and session_key and 'data_analyst_key' in dir() and data_analyst_key:
    requests.delete(f"{BASE_URL}/agents/{data_analyst_key}/sessions/{session_key}", headers=headers)
    print(f"Deleted previous session: {session_key}")
    session_key = None

if 'data_analyst_key' in dir() and data_analyst_key:
    resp = requests.delete(f"{BASE_URL}/agents/{data_analyst_key}", headers=headers)
    if resp.status_code == 204:
        print(f"Deleted previous agent: {data_analyst_key}")
    data_analyst_key = None

data_analyst_config = {
    "name": "Data Analyst",
    "description": "Agent specialized in data analysis using NumPy and Pandas-powered Lambda tools",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [
                {
                    "type": "inline",
                    "name": "data_analyst_instructions",
                    "template": """You are an expert data analyst with access to powerful data analysis tools.

Your capabilities:
1. **Statistical Analysis**: Use the statistical_analyzer tool to compute descriptive statistics, correlations, and percentiles
2. **Trend Analysis**: Use the trend_analyzer tool to identify patterns, growth rates, and trends in time-series data

When analyzing data:
1. First understand the data structure and what the user wants to learn
2. Choose the appropriate tool(s) for the analysis
3. Pass data as a JSON string to the tools
4. Interpret the results and explain insights in plain language
5. Suggest follow-up analyses if relevant

IMPORTANT:
- Always use tools for numerical computations - don't try to calculate statistics manually
- Explain what each metric means in the context of the user's data
- Highlight actionable insights and anomalies"""
                }
            ],
            "output_parser": {"type": "default"}
        }
    },
    "tool_configurations": {
        "statistical_analyzer": {
            "type": "lambda",
            "tool_id": statistical_analyzer_id
        },
        "trend_analyzer": {
            "type": "lambda",
            "tool_id": trend_analyzer_id
        }
    }
}

data_analyst_key = delete_and_create_agent(data_analyst_config, "Data Analyst")

Created agent 'Data Analyst' (key: agt_data_analyst_d71a)


## Step 4: Create a Session and Test the Agent

Let's create a session and test the data analyst agent with sample datasets.

In [6]:
# Delete previous session if re-running this cell
if 'session_key' in dir() and session_key:
    requests.delete(f"{BASE_URL}/agents/{data_analyst_key}/sessions/{session_key}", headers=headers)
    print(f"Deleted previous session: {session_key}")
    session_key = None

# Create a session
session_name = f"Data Analysis Demo {datetime.now().strftime('%Y%m%d-%H%M%S')}"
session_config = {
    "name": session_name,
    "metadata": {"purpose": "lambda_tools_demo"}
}

response = requests.post(
    f"{BASE_URL}/agents/{data_analyst_key}/sessions",
    headers=headers,
    json=session_config
)

if response.status_code == 201:
    session_data = response.json()
    session_key = session_data["key"]
    print(f"Session Created: {session_key}")
else:
    raise RuntimeError(f"Failed to create session: {response.status_code} - {response.text}")

Session Created: ase_data_analysis_demo_20260506-062133_91f1


In [7]:
# Helper function to chat with the agent
def chat_with_agent(agent_key, session_key, message, show_events=False):
    """Send a message to an agent and return the response."""
    message_data = {
        "messages": [{"type": "text", "content": message}],
        "stream_response": False
    }
    
    url = f"{BASE_URL}/agents/{agent_key}/sessions/{session_key}/events"
    response = requests.post(url, headers=headers, json=message_data)
    
    if response.status_code == 201:
        event_data = response.json()
        
        if show_events:
            print("\n------ Agent Events ------")
            for event in event_data.get('events', []):
                event_type = event.get('type', 'unknown')
                if event_type == 'tool_input':
                    tool_name = event.get('tool_configuration_name', 'N/A')
                    print(f"Tool Called: {tool_name}")
                    tool_input = event.get('tool_input', {})
                    # Show key parameters (not the full data)
                    params = {k: v[:100] + '...' if isinstance(v, str) and len(v) > 100 else v 
                              for k, v in tool_input.items() if k != 'data'}
                    if params:
                        print(f"  Params: {params}")
                elif event_type == 'tool_output':
                    tool_name = event.get('tool_configuration_name', 'N/A')
                    tool_output = event.get('tool_output', {})
                    output_str = json.dumps(tool_output)
                    if len(output_str) > 500:
                        output_str = output_str[:500] + '...'
                    print(f"  Output from {tool_name}: {output_str}")
            print("-" * 25 + "\n")
        
        # Extract agent output
        for event in event_data.get('events', []):
            if event.get('type') == 'agent_output':
                return event.get('content', 'No content')
        return "No agent output found"
    else:
        return f"Error: {response.status_code} - {response.text}"

### Example 1: Statistical Analysis

Let's analyze a sales dataset with the statistical analyzer.

In [8]:
# Sample sales data
sales_data = [
    {"product": "Widget A", "region": "North", "sales": 15000, "units": 150, "profit_margin": 0.25},
    {"product": "Widget A", "region": "South", "sales": 12000, "units": 120, "profit_margin": 0.22},
    {"product": "Widget A", "region": "East", "sales": 18000, "units": 180, "profit_margin": 0.28},
    {"product": "Widget A", "region": "West", "sales": 9000, "units": 90, "profit_margin": 0.20},
    {"product": "Widget B", "region": "North", "sales": 22000, "units": 110, "profit_margin": 0.35},
    {"product": "Widget B", "region": "South", "sales": 25000, "units": 125, "profit_margin": 0.38},
    {"product": "Widget B", "region": "East", "sales": 19000, "units": 95, "profit_margin": 0.32},
    {"product": "Widget B", "region": "West", "sales": 28000, "units": 140, "profit_margin": 0.40},
    {"product": "Widget C", "region": "North", "sales": 8000, "units": 200, "profit_margin": 0.15},
    {"product": "Widget C", "region": "South", "sales": 7500, "units": 188, "profit_margin": 0.14},
    {"product": "Widget C", "region": "East", "sales": 9500, "units": 238, "profit_margin": 0.18},
    {"product": "Widget C", "region": "West", "sales": 6000, "units": 150, "profit_margin": 0.12},
]

query = f"""I have sales data for three products across four regions. 
Please analyze this data and tell me:
1. Basic statistics for sales and profit margins
2. Correlation between sales, units, and profit margin

Here's the data:
{json.dumps(sales_data)}"""

print("User: Analyzing sales data...")
print("=" * 80)

response = chat_with_agent(data_analyst_key, session_key, query, show_events=True)
print(f"Agent Response:\n\n{response}")

User: Analyzing sales data...

------ Agent Events ------
Tool Called: statistical_analyzer
  Params: {'columns': 'sales,profit_margin', 'operations': 'describe'}
Tool Called: statistical_analyzer
  Params: {'columns': 'sales,units,profit_margin', 'operations': 'correlation'}
  Output from statistical_analyzer: {"success": true, "statistics": {"describe": {"sales": {"count": 12.0, "mean": 14916.666666666666, "std": 7412.622322563703, "min": 6000.0, "25%": 8750.0, "50%": 13500.0, "75%": 19750.0, "max": 28000.0}, "profit_margin": {"count": 12.0, "mean": 0.24916666666666668, "std": 0.09652680582317144, "min": 0.12, "25%": 0.1725, "50%": 0.235, "75%": 0.3275, "max": 0.4}}}, "metadata": {"rows": 12, "columns_analyzed": ["sales", "profit_margin"], "operations_performed": ["describe"]}}
  Output from statistical_analyzer: {"success": true, "statistics": {"correlation": {"sales": {"sales": 1.0, "units": -0.39664343302135424, "profit_margin": 0.9896428771138392}, "units": {"sales": -0.396643433

### Example 2: Trend Analysis

Let's analyze monthly revenue trends.

In [9]:
# Sample time-series data
revenue_data = [
    {"month": "2024-01-01", "revenue": 100000},
    {"month": "2024-02-01", "revenue": 105000},
    {"month": "2024-03-01", "revenue": 98000},
    {"month": "2024-04-01", "revenue": 112000},
    {"month": "2024-05-01", "revenue": 118000},
    {"month": "2024-06-01", "revenue": 125000},
    {"month": "2024-07-01", "revenue": 122000},
    {"month": "2024-08-01", "revenue": 135000},
    {"month": "2024-09-01", "revenue": 142000},
    {"month": "2024-10-01", "revenue": 138000},
    {"month": "2024-11-01", "revenue": 155000},
    {"month": "2024-12-01", "revenue": 168000},
]

query = f"""Analyze the revenue trend for this year. I want to know:
1. Is revenue trending up or down?
2. What's the growth rate?
3. What are the moving averages?

Here's the monthly data:
{json.dumps(revenue_data)}"""

print("User: Analyzing revenue trends...")
print("=" * 80)

response = chat_with_agent(data_analyst_key, session_key, query, show_events=True)
print(f"Agent Response:\n\n{response}")

User: Analyzing revenue trends...

------ Agent Events ------
Tool Called: trend_analyzer
  Params: {'date_column': 'month', 'value_column': 'revenue', 'analysis_type': 'all'}
  Output from trend_analyzer: {"success": true, "analysis": {"moving_averages": {"ma_3": {"latest": 153666.67, "values": [101000.0, 105000.0, 109333.33, 118333.33, 121666.67, 127333.33, 133000.0, 138333.33, 145000.0, 153666.67]}, "ma_7": {"latest": 140714.29, "values": [111428.57, 116428.57, 121714.29, 127428.57, 133571.43, 140714.29]}}, "growth_rates": {"period_over_period": {"latest": 0.0839, "mean": 0.0502}, "total_growth": 0.68}, "trend": {"slope": 5860.14, "intercept": 94269.23, "r_squared": 0.9334, "direction": "upward...
-------------------------

Agent Response:

Here's the analysis of your revenue trend for the year:

### Revenue Trend Analysis

1. **Trend Direction:**
   - The revenue is trending **upward**. The overall analysis indicates a positive trend with a high R-squared value of 0.9334, which sug

### Example 3: Multi-Tool Analysis

Let's ask for a comprehensive analysis that requires both tools.

In [10]:
# Comprehensive dataset
quarterly_data = [
    {"quarter": "2023-Q1", "date": "2023-01-01", "revenue": 450000, "costs": 320000, "customers": 1200},
    {"quarter": "2023-Q2", "date": "2023-04-01", "revenue": 480000, "costs": 335000, "customers": 1350},
    {"quarter": "2023-Q3", "date": "2023-07-01", "revenue": 520000, "costs": 360000, "customers": 1500},
    {"quarter": "2023-Q4", "date": "2023-10-01", "revenue": 610000, "costs": 400000, "customers": 1800},
    {"quarter": "2024-Q1", "date": "2024-01-01", "revenue": 580000, "costs": 390000, "customers": 1750},
    {"quarter": "2024-Q2", "date": "2024-04-01", "revenue": 650000, "costs": 420000, "customers": 2000},
    {"quarter": "2024-Q3", "date": "2024-07-01", "revenue": 720000, "costs": 450000, "customers": 2300},
    {"quarter": "2024-Q4", "date": "2024-10-01", "revenue": 800000, "costs": 480000, "customers": 2650},
]

query = f"""I need a comprehensive analysis of our quarterly business performance.

Please provide:
1. Statistical summary of revenue, costs, and customers
2. Correlation analysis - are revenue and customers correlated?
3. Trend analysis on revenue - what's our growth trajectory?

Here's the data:
{json.dumps(quarterly_data)}"""

print("User: Comprehensive business analysis...")
print("=" * 80)

response = chat_with_agent(data_analyst_key, session_key, query, show_events=True)
print(f"Agent Response:\n\n{response}")

User: Comprehensive business analysis...

------ Agent Events ------
Tool Called: statistical_analyzer
  Params: {'columns': 'revenue,costs,customers', 'operations': 'describe'}
Tool Called: statistical_analyzer
  Params: {'columns': 'revenue,customers', 'operations': 'correlation'}
Tool Called: trend_analyzer
  Params: {'date_column': 'date', 'value_column': 'revenue', 'analysis_type': 'all'}
  Output from statistical_analyzer: {"success": true, "statistics": {"describe": {"revenue": {"count": 8.0, "mean": 601250.0, "std": 119933.01701962046, "min": 450000.0, "25%": 510000.0, "50%": 595000.0, "75%": 667500.0, "max": 800000.0}, "costs": {"count": 8.0, "mean": 394375.0, "std": 55255.09026325086, "min": 320000.0, "25%": 353750.0, "50%": 395000.0, "75%": 427500.0, "max": 480000.0}, "customers": {"count": 8.0, "mean": 1818.75, "std": 487.66023007827897, "min": 1200.0, "25%": 1462.5, "50%": 1775.0, "75%": 2075.0, "max": 2650...
  Output from statistical_analyzer: {"success": true, "statisti

## Cleanup (Optional)

Delete the resources created in this notebook.

In [11]:
import time

# Delete the agent. The server's DELETE is asynchronous — /tools DELETE will
# reject with 409 "being used in different configurations" until the agent's
# reference to the tools is fully torn down, so we poll until the agent is
# actually gone before deleting the tools.
if data_analyst_key:
    response = requests.delete(f"{BASE_URL}/agents/{data_analyst_key}", headers=headers)
    if response.status_code == 204:
        print(f"Deleted agent: {data_analyst_key}")
    else:
        print(f"Error deleting agent: {response.text}")

    deadline = time.time() + 30
    while time.time() < deadline:
        check = requests.get(f"{BASE_URL}/agents/{data_analyst_key}", headers=headers)
        if check.status_code == 404:
            break
        time.sleep(2)

# Delete the Lambda tools, retrying on 409 in case the agent reference
# hasn't fully cleared yet.
for tool_id, tool_name in [
    (statistical_analyzer_id, "statistical_analyzer"),
    (trend_analyzer_id, "trend_analyzer"),
]:
    if not tool_id:
        continue
    for attempt in range(4):
        response = requests.delete(f"{BASE_URL}/tools/{tool_id}", headers=headers)
        if response.status_code == 204:
            print(f"Deleted tool: {tool_name} ({tool_id})")
            break
        if response.status_code == 409 and attempt < 3:
            time.sleep(2 ** attempt)
            continue
        print(f"Error deleting {tool_name}: {response.text}")
        break

Deleted agent: agt_data_analyst_d71a
Deleted tool: statistical_analyzer (tol_6330)
Deleted tool: trend_analyzer (tol_6331)
